# LanceDB image-index benchmark

A performance experiment, separate from publication figure generation. Read the backend's configured Lance table and cosine embedding searches; build candidate indexes only on a fresh full-table copy under `analyses/results/indexing/`.

This tests vector retrieval, not FastAPI latency or embedding-model inference. A run copies all source rows and needs enough disk for the copy and indexes. The source table is pinned to its current version. Existing backend indexes are never reset, replaced, or selected for deployment.

Install with `uv sync --project analyses --extra indexing --locked`. See [benchmark instructions](../README.md#indexing-and-backend-performance).

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "backend/app/configs/config.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from analyses.benchmarks.lance_indexing import (
    BenchmarkConfig,
    load_source,
    open_source,
    run_benchmark,
    validate_source,
)

source_settings = load_source(ROOT)
source_table = open_source(source_settings)
print(
    f"Source: {source_settings.table}; version {source_table.version}; "
    f"{source_table.count_rows():,} images"
)
source_table.list_indices()

## Experiment configuration

Both embedding columns use cosine distance for the exact reference and ANN queries, matching backend retrieval. The same seeded query sample is used for every candidate. Samples are spread across the full table, not taken from its first five rows.

`None` for search controls preserves LanceDB defaults used by the backend. Configure partitions/subvectors for the dataset: PQ subvector counts must divide the vector dimension. Warmup queries and index-building time are excluded from search latency; build time is reported separately.

The exact baseline bypasses vector indexes. There is no duplicate “flat” candidate. Query images remain in the searched collection, so recall includes self matches. Repeated sequential searches describe warm-cache behavior, not concurrent throughput or cold starts.

In [ ]:
config = BenchmarkConfig(
    query_count=100,
    repeats=5,
    warmup_queries=10,
    top_k=10,
    seed=42,
    num_partitions=256,
    pq_sub_vectors=128,
    hnsw_pq_sub_vectors=64,
    nprobes=None,
    refine_factor=None,
    index_types=("IVF_PQ", "IVF_HNSW_SQ", "IVF_HNSW_PQ"),
    vector_columns=("unicom_embeddings", "clip_embeddings"),
    recall_threshold=0.95,
)
validate_source(source_table, config)

## Run on an isolated copy

Each execution creates a unique retained run directory. Index builds and replacements target only that run's scratch table. Failed experiments record the error and preserve completed measurements; failures do not silently become successful rows.

The run exports summary CSV, per-query timings, recommendations, and configuration/provenance JSON. It does not generate figures or overwrite `analyses/data/indexing_benchmark.csv`.

In [ ]:
results, recommendations, run_directory = run_benchmark(source_settings, config)
print(f"Benchmark artifacts: {run_directory}")
results

## Compare candidates

Recommendations select the fastest measured candidate meeting the configured recall threshold separately for each embedding column. They may select the exact baseline. They are advisory: no backend index is applied automatically. QPS is derived from sequential query durations, not a load test.

For a publication figure, independently review a completed CSV before selecting it as plotting input. Scratch databases are retained for inspection and may consume substantial disk space.

In [ ]:
recommendations